In [2]:
import numpy as np
import keras
import keras_tuner as kt

2026-02-20 16:28:56.954004: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-20 16:28:58.613509: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-20 16:29:42.403411: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Спочатку спробую спочатку варіацію з прикладу

In [3]:
max_features = 20000
maxlen = 200

In [13]:
(x_train, y_train), (x_val, y_val) = keras.datasets.imdb.load_data(num_words=max_features)

/mnt/c/VS Code project/course_ml/.venv/lib/python3.12/site-packages/numpy/lib/_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


In [14]:
inputs = keras.Input(shape=(None,), dtype="int32")
x = keras.layers.Embedding(max_features, 128)(inputs)
x = keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True))(x)
x = keras.layers.Bidirectional(keras.layers.LSTM(64))(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, None, 128)      │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, None, 128)      │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,757,761 (10.52 MB)

 Trainable params: 2,757,761 (10.52 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
x_train = keras.utils.pad_sequences(x_train, maxlen=maxlen)
x_val = keras.utils.pad_sequences(x_val, maxlen=maxlen)

In [16]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, batch_size=32, epochs=2, validation_data=(x_val, y_val))

Epoch 1/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 82s 101ms/step - accuracy: 0.8076 - loss: 0.4206 - val_accuracy: 0.8601 - val_loss: 0.3471
Epoch 2/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 69s 89ms/step - accuracy: 0.9051 - loss: 0.2448 - val_accuracy: 0.8569 - val_loss: 0.3542


При цій комбінації точність дорівнює при тренуванні 0.9086 та зайняло це 2 хв 30 с. Валідація дала 0.8569 точність 

Далі буду эксперементувати, спробую зробити ще одну епоху 

In [18]:
model.fit(x_train, y_train, batch_size=32, epochs=1, validation_data=(x_val, y_val))

782/782 ━━━━━━━━━━━━━━━━━━━━ 65s 83ms/step - accuracy: 0.9479 - loss: 0.1443 - val_accuracy: 0.8643 - val_loss: 0.4040


При тренуванні модель навчилась до 0.9479. Валідація дала 0.8643, тобто різниця зовсім невелека

Спробую зробити теж саме, але зменшити max_features

In [22]:
max_features = 5000

In [23]:
(x_train, y_train), (x_val, y_val) = keras.datasets.imdb.load_data(num_words=max_features)
inputs = keras.Input(shape=(None,), dtype="int32")
x = keras.layers.Embedding(max_features, 128)(inputs)
x = keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True))(x)
x = keras.layers.Bidirectional(keras.layers.LSTM(64))(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
x_train = keras.utils.pad_sequences(x_train, maxlen=maxlen)
x_val = keras.utils.pad_sequences(x_val, maxlen=maxlen)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, batch_size=32, epochs=2, validation_data=(x_val, y_val))

Epoch 1/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 77s 94ms/step - accuracy: 0.8024 - loss: 0.4250 - val_accuracy: 0.8653 - val_loss: 0.3273
Epoch 2/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 66s 84ms/step - accuracy: 0.8884 - loss: 0.2747 - val_accuracy: 0.8374 - val_loss: 0.3758


При зменшені кількості фічей у 4 рази зменшилась точність приблизно на 2 відсотки. Ще порівняю одну епоху

In [24]:
model.fit(x_train, y_train, batch_size=32, epochs=1, validation_data=(x_val, y_val))

782/782 ━━━━━━━━━━━━━━━━━━━━ 65s 83ms/step - accuracy: 0.8637 - loss: 0.3132 - val_accuracy: 0.8450 - val_loss: 0.3617


На третій епосі модель почала перетроновувати, вже фічей недостатньо, проте валідація трошки виросла 

Спробую залишити 5000 фічей, але прибрати maxlen, тобто усі строки будуть мати такий же розмір, як і найбільша строка.

In [25]:
(x_train, y_train), (x_val, y_val) = keras.datasets.imdb.load_data(num_words=max_features)
inputs = keras.Input(shape=(None,), dtype="int32")
x = keras.layers.Embedding(max_features, 128)(inputs)
x = keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True))(x)
x = keras.layers.Bidirectional(keras.layers.LSTM(64))(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
x_train = keras.utils.pad_sequences(x_train)
x_val = keras.utils.pad_sequences(x_val)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, batch_size=32, epochs=2, validation_data=(x_val, y_val))

/mnt/c/VS Code project/course_ml/.venv/lib/python3.12/site-packages/numpy/lib/_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


Epoch 1/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 658s 838ms/step - accuracy: 0.7644 - loss: 0.4843 - val_accuracy: 0.8414 - val_loss: 0.3740
Epoch 2/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 647s 827ms/step - accuracy: 0.8448 - loss: 0.3583 - val_accuracy: 0.8710 - val_loss: 0.2959


Модель тренувалася аж 22 хвилини та видала +- такий же результат

Тепер навпаки обрізати maxlen на 100 та 50

In [9]:
for max_len in [100, 50]:
    (x_train, y_train), (x_val, y_val) = keras.datasets.imdb.load_data(num_words=max_features)
    inputs = keras.Input(shape=(None,), dtype="int32")
    x = keras.layers.Embedding(max_features, 128)(inputs)
    x = keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True))(x)
    x = keras.layers.Bidirectional(keras.layers.LSTM(64))(x)
    outputs = keras.layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs)
    x_train = keras.utils.pad_sequences(x_train, maxlen=max_len)
    x_val = keras.utils.pad_sequences(x_val, maxlen=max_len)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(x_train, y_train, batch_size=32, epochs=2, validation_data=(x_val, y_val))

/mnt/c/VS Code project/course_ml/.venv/lib/python3.12/site-packages/numpy/lib/_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)
I0000 00:00:1771585570.927296     913 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1767 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Epoch 1/2


2026-02-20 12:06:17.155535: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900


782/782 ━━━━━━━━━━━━━━━━━━━━ 58s 67ms/step - accuracy: 0.8168 - loss: 0.3930 - val_accuracy: 0.8357 - val_loss: 0.3594
Epoch 2/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 54s 69ms/step - accuracy: 0.9182 - loss: 0.2076 - val_accuracy: 0.8422 - val_loss: 0.4611
Epoch 1/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 41s 49ms/step - accuracy: 0.7820 - loss: 0.4494 - val_accuracy: 0.8189 - val_loss: 0.4051
Epoch 2/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 31s 40ms/step - accuracy: 0.8922 - loss: 0.2665 - val_accuracy: 0.8162 - val_loss: 0.4227


Модель натренувалася швидше та отримуємо таку статистику при max_features = 5000
Maxlen:
* 200 - test 0.8884, val 0.8374 
* 100 - test 0.9182, val 0.8422 
* 50 - test 0.8922, val 0.8162 

Бачимо, що при 100 точність виросла, а при 50 впала, тому для цього датасету найкраще ставити 100, адже це швидше та точніше

Спробую погратися з кількістю шарів та нейронів

In [11]:
max_features = 5000
maxlen = 100
(x_train, y_train), (x_val, y_val) = keras.datasets.imdb.load_data(num_words=max_features)
inputs = keras.Input(shape=(None,), dtype="int32")
x = keras.layers.Embedding(max_features, 128)(inputs)
x = keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True))(x)
x = keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True))(x)
x = keras.layers.Bidirectional(keras.layers.LSTM(64))(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
x_train = keras.utils.pad_sequences(x_train, maxlen=maxlen)
x_val = keras.utils.pad_sequences(x_val, maxlen=maxlen)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, batch_size=32, epochs=2, validation_data=(x_val, y_val))

/mnt/c/VS Code project/course_ml/.venv/lib/python3.12/site-packages/numpy/lib/_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


Epoch 1/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 68s 83ms/step - accuracy: 0.8028 - loss: 0.4135 - val_accuracy: 0.8542 - val_loss: 0.3316
Epoch 2/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 62s 79ms/step - accuracy: 0.8813 - loss: 0.2832 - val_accuracy: 0.8587 - val_loss: 0.3202


Додавши ще один шар з 64(128) нейронів модель показала гірший результат при тренуванні та трішки кращий при валідації

In [12]:
max_features = 20000
maxlen = 200
(x_train, y_train), (x_val, y_val) = keras.datasets.imdb.load_data(num_words=max_features)
inputs = keras.Input(shape=(None,), dtype="int32")
x = keras.layers.Embedding(max_features, 128)(inputs)
x = keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True))(x)
x = keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True))(x)
x = keras.layers.Bidirectional(keras.layers.LSTM(64))(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
x_train = keras.utils.pad_sequences(x_train, maxlen=maxlen)
x_val = keras.utils.pad_sequences(x_val, maxlen=maxlen)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, batch_size=32, epochs=2, validation_data=(x_val, y_val))

Epoch 1/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 105s 129ms/step - accuracy: 0.8114 - loss: 0.4195 - val_accuracy: 0.8626 - val_loss: 0.3268
Epoch 2/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 96s 122ms/step - accuracy: 0.9065 - loss: 0.2440 - val_accuracy: 0.8585 - val_loss: 0.3333


In [13]:
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_5 (Embedding)         │ (None, None, 128)      │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_11                │ (None, None, 128)      │        98,816 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_12                │ (None, None, 128)      │        98,816 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_13                │ (None, 128)            │        98,816 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,569,733 (32.69 MB)

 Trainable params: 2,856,577 (10.90 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,713,156 (21.79 MB)

Якщо брати комбінацію max_features = 20000, maxlen = 200, то модель видала майже однаковиий результат, проти важить у 2 рази більше

Спробую залишити 2 шари, але збільшити кількість нейронів

In [14]:
max_features = 5000
maxlen = 100
(x_train, y_train), (x_val, y_val) = keras.datasets.imdb.load_data(num_words=max_features)
inputs = keras.Input(shape=(None,), dtype="int32")
x = keras.layers.Embedding(max_features, 128)(inputs)
x = keras.layers.Bidirectional(keras.layers.LSTM(128, return_sequences=True))(x)
x = keras.layers.Bidirectional(keras.layers.LSTM(128))(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
x_train = keras.utils.pad_sequences(x_train, maxlen=maxlen)
x_val = keras.utils.pad_sequences(x_val, maxlen=maxlen)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, batch_size=32, epochs=2, validation_data=(x_val, y_val))

Epoch 1/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 52s 63ms/step - accuracy: 0.7994 - loss: 0.4292 - val_accuracy: 0.8250 - val_loss: 0.3832
Epoch 2/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 50s 64ms/step - accuracy: 0.8773 - loss: 0.2940 - val_accuracy: 0.8470 - val_loss: 0.3457


Тестувальна точність трохи впала, валідаційна трохи зросла, швидкість трохи впала

In [15]:
(x_train, y_train), (x_val, y_val) = keras.datasets.imdb.load_data(num_words=max_features)
inputs = keras.Input(shape=(None,), dtype="int32")
x = keras.layers.Embedding(max_features, 128)(inputs)
x = keras.layers.Bidirectional(keras.layers.LSTM(32, return_sequences=True))(x)
x = keras.layers.Bidirectional(keras.layers.LSTM(32))(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
x_train = keras.utils.pad_sequences(x_train, maxlen=maxlen)
x_val = keras.utils.pad_sequences(x_val, maxlen=maxlen)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, batch_size=32, epochs=2, validation_data=(x_val, y_val))

Epoch 1/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 48s 57ms/step - accuracy: 0.8077 - loss: 0.4089 - val_accuracy: 0.8470 - val_loss: 0.3398
Epoch 2/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 46s 59ms/step - accuracy: 0.8874 - loss: 0.2730 - val_accuracy: 0.8581 - val_loss: 0.3403


Тестувальна впала, але валідаційна трохи зросла

Я знайшов таку річ як keras_tuner та спробую за допомогою нього підібрати кількість нейронів

In [4]:
def build_model_20000(hp):
    inputs = keras.layers.Input(shape=(None,), name="int32")
    x = keras.layers.Embedding(20000, 128)(inputs)
    
    hp_units = hp.Int('units', min_value=32, max_value=256, step=32)
    
    x = keras.layers.Bidirectional(keras.layers.LSTM(units=hp_units, return_sequences=True))(x)
    x = keras.layers.Bidirectional(keras.layers.LSTM(units=hp_units))(x)
    outputs = keras.layers.Dense(1, activation='sigmoid', name="dense")(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

In [18]:
def build_model_5000(hp):
    inputs = keras.layers.Input(shape=(None,), name="int32")
    x = keras.layers.Embedding(5000, 128)(inputs)
    
    hp_units = hp.Int('units', min_value=32, max_value=256, step=32)
    
    x = keras.layers.Bidirectional(keras.layers.LSTM(units=hp_units, return_sequences=True))(x)
    x = keras.layers.Bidirectional(keras.layers.LSTM(units=hp_units))(x)
    outputs = keras.layers.Dense(1, activation='sigmoid', name="dense")(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

Спочатку підберу для 20000 фічей

In [5]:
(x_train, y_train), (x_val, y_val) = keras.datasets.imdb.load_data(num_words=20000)
x_train = keras.utils.pad_sequences(x_train, maxlen=200)
x_val = keras.utils.pad_sequences(x_val, maxlen=200)

tuner = kt.RandomSearch(
    build_model_20000,
    objective='val_accuracy',
    max_trials=5,
    directory='tuning_dir',
    project_name='nlp_model_tuning_20000'
)

tuner.search(x_train, y_train, epochs=2, validation_data=(x_val, y_val))

Trial 5 Complete [00h 02m 49s]
val_accuracy: 0.8712400197982788

Best val_accuracy So Far: 0.8712400197982788
Total elapsed time: 00h 14m 39s


In [9]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best units:",best_hps.get('units'))
tuner.results_summary()

Best units: 160
Results summary
Results in tuning_dir/nlp_model_tuning_20000
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 4 summary
Hyperparameters:
units: 160
Score: 0.8712400197982788

Trial 0 summary
Hyperparameters:
units: 32
Score: 0.866599977016449

Trial 2 summary
Hyperparameters:
units: 224
Score: 0.8614000082015991

Trial 1 summary
Hyperparameters:
units: 256
Score: 0.855679988861084

Trial 3 summary
Hyperparameters:
units: 128
Score: 0.8149200081825256


Найкраща точність у моделі з 160 нейронами, проте різниця невелика, до 2%, крім 128, там різнаця 6%

In [ ]:
(x_train, y_train), (x_val, y_val) = keras.datasets.imdb.load_data(num_words=5000)
x_train = keras.utils.pad_sequences(x_train, maxlen=200)
x_val = keras.utils.pad_sequences(x_val, maxlen=200)
y_train = np.array(y_train, dtype='int32')
y_val = np.array(y_val, dtype='int32')

tuner_5000 = kt.RandomSearch(
    build_model_5000,
    objective='val_accuracy',
    max_trials=5,
    directory='tuning_dir',
    project_name='nlp_model_tuning_5000'
)

tuner_5000.search(x_train, y_train, epochs=2, validation_data=(x_val, y_val))

Trial 5 Complete [00h 02m 40s]
val_accuracy: 0.859279990196228

Best val_accuracy So Far: 0.8675199747085571
Total elapsed time: 00h 13m 27s


In [ ]:
best_hps = tuner_5000.get_best_hyperparameters(num_trials=1)[0]
best_hps.get('units')

32

Тут найкраща модель з 32 нейронами, але знову ж таки, різниця невелика